In [ ]:
import sys
import torch

# ---------------------------------------------------------
# Configuration and Paths
# ---------------------------------------------------------
PROJECT_ROOT = '/home/andrew/Andrew/Fishial2402/fish-identification'
sys.path.append(PROJECT_ROOT)

# Import custom modules after appending the project root to sys.path
from module.classification_package.src.model_v2 import StableEmbeddingModelViT

# Grouping all hardcoded variables into a single configuration block
CONFIG = {
    "model_ckpt": "/home/andrew/Andrew/Fishial2402/Experiments/v18_866_classes/vit_base_patch14_reg4_dinov2.lvd142m_20260331_120818/checkpoints/model-epoch88-acc9322.ckpt",
    "db_path": "/home/andrew/Andrew/Fishial2402/Experiments/v18_866_classes/vit_base_patch14_reg4_dinov2.lvd142m_20260331_120818/natural_gallery_K3_model-epoch88-acc9322.pt",
    "export_save_path": "model_bundle.pt",
    "img_size": (154, 434), # Dimensions: (Height, Width)
    "device": "cpu",        # Use "cuda" if you intend to export and run on GPU
    "embedding_dim": 768,
    "num_classes": 866
}


def export_model(config: dict) -> None:
    """
    Loads the PyTorch Lightning checkpoint and exports it to TorchScript.
    """
    print(f"Loading model from checkpoint to {config['device'].upper()}...")
    
    # Load the model directly to the target device
    model = StableEmbeddingModelViT.load_from_checkpoint(
        checkpoint_path=config["model_ckpt"],
        map_location=config["device"],
        strict=True
    )
    model.eval()

    print("Exporting model to TorchScript...")
    # Export the model bundle
    model.export_to_torchscript(
        save_path=config["export_save_path"],
        img_size=config["img_size"],   
        device=config["device"],          
        natural_gallery_path=config["db_path"],
        embed_natural_centroids=True,
        embed_arcface=True,
    )
    print(f"Model successfully exported to: {config['export_save_path']}\n")


def test_exported_model(config: dict) -> None:
    """
    Loads the exported TorchScript model and tests it with various batch sizes.
    """
    print(f"Loading exported TorchScript model from {config['export_save_path']}...")
    
    # Ensure we load the exact file we just saved
    engine = torch.jit.load(config["export_save_path"])
    engine.eval()

    # Ensure dynamic batching works at runtime
    print("Testing dynamic batch sizes...")
    with torch.no_grad():
        for bs in [1, 4, 16, 32]:
            # Generate dummy input tensor
            x = torch.randn(bs, 3, config["img_size"][0], config["img_size"][1])
            
            # Run inference
            emb, logits = engine(x)

            # Validate output shapes
            assert emb.shape == (bs, config["embedding_dim"]), f"Embedding shape failed at batch_size={bs}"
            assert logits.shape == (bs, config["num_classes"]), f"Logits shape failed at batch_size={bs}"
            
            print(f"Batch Size: {bs:>2} | ✅ | Embedding shape: {tuple(emb.shape)} | Logits shape: {tuple(logits.shape)}")


if __name__ == "__main__":
    export_model(CONFIG)
    test_exported_model(CONFIG)